# Muhtemel Ask - 03 ARCHIVE & CLEANUP
Run this notebook only after `02_FINALIZE.ipynb` reports **FINALIZATION PASS**. It creates or revalidates a high-quality soft-sub MKV by copying the compressed video/audio streams without re-encoding. Indonesian is the default subtitle and Turkish is the second track.

Cleanup is deliberately off by default. When enabled, it removes the episode's ASR/audio, schema, translation ZIPs, review workbook, markers and reports only after the MKV, both embedded subtitles and A/V stream hashes pass verification. Drive needs about one source-video-sized block of temporary free space while the atomic MKV is being created (roughly 0.5 GB for Episode 11).

In [ ]:
EPISODE = 11  # @param {type:"integer"}
DELETE_INTERMEDIATES = False  # @param {type:"boolean"}
KEEP_SOURCE_VIDEO = False  # @param {type:"boolean"}

if isinstance(EPISODE, bool) or not isinstance(EPISODE, int) or EPISODE < 1:
    raise ValueError("EPISODE must be a positive integer")
for _name, _value in {
    "DELETE_INTERMEDIATES": DELETE_INTERMEDIATES,
    "KEEP_SOURCE_VIDEO": KEEP_SOURCE_VIDEO,
}.items():
    if not isinstance(_value, bool):
        raise ValueError(f"{_name} must be True or False")

## Mount Drive and prepare FFmpeg

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import importlib
import shutil
import subprocess
import sys

SYSTEM_ROOT = Path("/content/drive/MyDrive/Muhtemel_Ask_Subtitles/SYSTEM")
if not (SYSTEM_ROOT / "src/episode_archive.py").is_file():
    raise FileNotFoundError(
        f"Archive module was not found at {SYSTEM_ROOT}. Upload the current SYSTEM release."
    )
if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
if str(SYSTEM_ROOT) not in sys.path:
    sys.path.insert(0, str(SYSTEM_ROOT))
for _module_name in list(sys.modules):
    if _module_name == "src" or _module_name.startswith("src."):
        del sys.modules[_module_name]
importlib.invalidate_caches()
print("Runtime ready.")

## Resolve the exact episode and verification receipt
The receipt is stored outside the episode workspace so archived MKV/SRT files remain verifiable after intermediate reports are removed.

In [ ]:
import src.episode_archive as episode_archive

_module_path = Path(episode_archive.__file__).resolve()
if SYSTEM_ROOT.resolve() not in _module_path.parents:
    raise RuntimeError(f"Loaded archive module from an unexpected path: {_module_path}")
archive_episode = episode_archive.archive_episode

EPISODE_NAME = f"Muhtemel Ask {EPISODE}.Bolum"
DRIVE_ROOT = Path("/content/drive/MyDrive/Muhtemel_Ask_Subtitles")
EPISODE_ROOT = DRIVE_ROOT / "EPISODES" / EPISODE_NAME
RECEIPT_PATH = DRIVE_ROOT / "ARCHIVE_REPORTS" / f"{EPISODE_NAME}_ARCHIVE_RECEIPT.json"
if not EPISODE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Episode workspace not found: {EPISODE_ROOT}. Run 01_PREPARE and 02_FINALIZE first."
    )
print(f"Episode: {EPISODE_ROOT}")
print(f"Receipt: {RECEIPT_PATH}")

## Build/verify MKV, then preview or perform cleanup
With `DELETE_INTERMEDIATES = False`, this is a dry run: the verified MKV is kept and the exact cleanup plan is shown, but nothing is deleted. With cleanup enabled, type the exact confirmation phrase when prompted.

In [ ]:
def request_cleanup_confirmation(expected):
    print("WARNING: cleanup removes PREPARE/FINALIZE resumability for this episode.")
    print("The verified MKV and all SRT files will remain" +
          (" together with the source video." if KEEP_SOURCE_VIDEO else "."))
    print("Any unexpected extra video is also kept and reported; it is never auto-deleted.")
    return input(f"Type exactly '{expected}' to continue: ")

archive_result = archive_episode(
    episode_root=EPISODE_ROOT,
    receipt_path=RECEIPT_PATH,
    episode=EPISODE,
    delete_intermediates=DELETE_INTERMEDIATES,
    keep_source_video=KEEP_SOURCE_VIDEO,
    confirmation=request_cleanup_confirmation if DELETE_INTERMEDIATES else None,
)

## Result
Files moved to Google Drive Trash still count toward quota. Inspect Trash and empty it manually if you want storage released immediately; this notebook never empties unrelated Drive Trash.

In [ ]:
from pprint import pprint

pprint(archive_result, sort_dicts=False)
print("")
print("ARCHIVE WORKFLOW PASS")
if not DELETE_INTERMEDIATES:
    print("Dry run only: set DELETE_INTERMEDIATES=True to reclaim the previewed space.")
else:
    print("Cleanup completed. Inspect and empty Google Drive Trash to release quota.")